In [ ]:
"""
Circular Volcano Plot
=======================
- Red/blue color intensity mapped to -log10(ivw_fdr): more significant = deeper
- Point size mapped to |ivw_beta|
- Point shape mapped to source subtype
- NPG color palette, cell type labels in center band
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.colorbar
import math, os


# ========================= Configuration =========================

DATA_DIR   = "./data/volcano/"     # Directory containing {celltype}_volcano_merged.csv files
OUTPUT_DIR = "./results/"     # Output directory for figures

FDR_THRESHOLD = 0.05
FDR_CAP       = 50       # Cap for -log10(FDR) display
MAX_SIG       = 800      # Max significant points to plot per cell type
MAX_NONSIG    = 400      # Max non-significant points to plot
TOP_N         = 5        # Top genes to label per direction

DOT_MIN  = 15
DOT_MAX  = 120
BETA_CAP = 30

BG_COLOR     = "#F5F5F5"
BG_ALPHA     = 0.45
NONSIG_COLOR = "#E0E0E0"
NONSIG_ALPHA = 0.12

FIG_SIZE = (18, 18)
DPI      = 300
GAP_DEG  = 3

MARKERS = ["o", "s", "^", "D", "v", "P", "X", "p", "h", "*"]

# Cell types and NPG colors
cell_types = ["Ast", "End", "Ext", "IN", "MG", "MiGA", "OD", "OPC"]
NPG = ["#E64B35", "#4DBBD5", "#00A087", "#3C5488",
       "#F39B7F", "#8491B4", "#91D1C2", "#DC0000"]
ctc = {ct: NPG[i] for i, ct in enumerate(cell_types)}

# Color gradients: light -> dark
up_cmap = LinearSegmentedColormap.from_list("up", ["#FADBD8", "#E64B35", "#922B21"])
dn_cmap = LinearSegmentedColormap.from_list("dn", ["#D4EFDF", "#4DBBD5", "#1A5276"])


# ========================= Helpers =========================

def size_map(ab):
    """Map |beta| to dot size."""
    return DOT_MIN + (DOT_MAX - DOT_MIN) * (min(ab, BETA_CAP) / BETA_CAP)


def color_by_fdr(nlf_capped, cap, direction):
    """Map -log10(FDR) to color intensity. Higher = deeper color."""
    intensity = min(nlf_capped / cap, 1.0)
    intensity = 0.15 + intensity * 0.85  # Floor at 0.15 to avoid near-white
    if direction == "\u2191":
        return up_cmap(intensity)
    else:
        return dn_cmap(intensity)


# ========================= Load Data =========================

all_data = {}
for ct in cell_types:
    fp = os.path.join(DATA_DIR, f"{ct}_volcano_merged.csv")
    if not os.path.exists(fp):
        print(f"Warning: {fp} not found"); continue
    df = pd.read_csv(fp)
    df["nlf"] = df["ivw_fdr"].apply(lambda x: -math.log10(x) if x > 0 else 300)
    df["ab"] = df["ivw_beta"].abs()
    df["sig"] = df["ivw_fdr"] < FDR_THRESHOLD
    df["nc"] = df["nlf"].clip(upper=FDR_CAP)

    sources = sorted(df["source"].unique())
    sm_map = {src: MARKERS[j % len(MARKERS)] for j, src in enumerate(sources)}

    sig = df[df["sig"]]; ns = df[~df["sig"]]
    if len(sig) > MAX_SIG: sig = sig.sample(MAX_SIG, random_state=42)
    if len(ns) > MAX_NONSIG: ns = ns.sample(MAX_NONSIG, random_state=42)
    pdf = pd.concat([sig, ns])

    tu = df[df["direction"] == "\u2191"].nlargest(TOP_N, "ab")[
        ["gene", "nc", "ab", "source"]].to_dict("records")
    td = df[df["direction"] == "\u2193"].nlargest(TOP_N, "ab")[
        ["gene", "nc", "ab", "source"]].to_dict("records")

    all_data[ct] = {"pdf": pdf, "tu": tu, "td": td,
                    "sources": sources, "sm": sm_map}
    print(f"  {ct}: {len(pdf)} points, subtypes={sources}")


# ========================= Plot =========================

fig, ax = plt.subplots(figsize=FIG_SIZE, subplot_kw={"projection": "polar"})
fig.patch.set_facecolor("white")

n = len(cell_types)
gap_r = np.radians(GAP_DEG)
sec_r = (2 * np.pi - n * gap_r) / n

di, bi, bo, do = 0.28, 0.62, 0.70, 1.05
up_r = do - bo
dn_r = bi - di

np.random.seed(42)

for i, ct in enumerate(cell_types):
    if ct not in all_data:
        continue
    ss = i * (sec_r + gap_r) - np.pi / 2
    se = ss + sec_r
    smid = (ss + se) / 2
    d = all_data[ct]

    # Background arcs
    th = np.linspace(ss, se, 120)
    ax.fill_between(th, bo, do, color=BG_COLOR, alpha=BG_ALPHA, zorder=0)
    ax.fill_between(th, di, bi, color=BG_COLOR, alpha=BG_ALPHA, zorder=0)

    # Center color band
    ax.fill_between(np.linspace(ss, se, 80), bi, bo,
                    color=ctc[ct], alpha=0.92, zorder=5)

    # Cell type label
    adeg = np.degrees(smid) % 360
    txt_rot = adeg - 90 if 0 <= adeg <= 180 else adeg + 90
    ax.text(smid, (bi + bo) / 2, ct, ha="center", va="center",
            fontsize=11, fontweight="bold", color="white",
            rotation=txt_rot, rotation_mode="anchor", zorder=6)

    # Scatter points with FDR-mapped color intensity
    for src in d["sources"]:
        sub = d["pdf"][d["pdf"]["source"] == src]
        mk = d["sm"][src]

        for _, row in sub.iterrows():
            t = np.random.uniform(ss + 0.008, se - 0.008)
            ny = row["nc"] / FDR_CAP
            ds = size_map(row["ab"])

            if row["direction"] == "\u2191":
                r = bo + 0.01 + ny * up_r * 0.90
            else:
                r = bi - 0.01 - ny * dn_r * 0.90

            if row["sig"]:
                c = color_by_fdr(row["nc"], FDR_CAP, row["direction"])
                a = 0.7
            else:
                c = NONSIG_COLOR
                a = NONSIG_ALPHA

            ax.scatter(t, r, s=ds, c=[c], alpha=a, marker=mk,
                       edgecolors="none", zorder=2)

    # Upregulated gene labels (outer ring)
    lrs_up = do + 0.03
    for j, g in enumerate(d["tu"][:TOP_N]):
        t = se - sec_r * (0.1 + 0.15 * j)
        ny = g["nc"] / FDR_CAP
        gr = bo + 0.01 + ny * up_r * 0.90
        lr = lrs_up + j * 0.07
        mk = d["sm"].get(g["source"], "o")
        gc = color_by_fdr(g["nc"], FDR_CAP, "\u2191")

        ax.scatter(t, min(gr, do - 0.01), s=size_map(g["ab"]),
                   c=[gc], alpha=0.95, marker=mk,
                   edgecolors="white", linewidth=0.5, zorder=6)
        ax.plot([t, t], [min(gr, do), lr - 0.01],
                color="#AAA", lw=0.4, alpha=0.5, zorder=4)
        ad2 = np.degrees(t) % 360
        tr = ad2 - 90 if ad2 <= 180 else ad2 + 90
        h2 = "left" if ad2 <= 180 else "right"
        ax.text(t, lr, g["gene"], fontsize=5.5, color="#333",
                fontstyle="italic", rotation=tr,
                rotation_mode="anchor", ha=h2, va="center", zorder=10)

    # Downregulated gene labels (inner ring)
    lrs_dn = di - 0.02
    for j, g in enumerate(d["td"][:TOP_N]):
        t = ss + sec_r * (0.08 + 0.16 * j)
        ny = g["nc"] / FDR_CAP
        gr = bi - 0.01 - ny * dn_r * 0.90
        lr = lrs_dn - j * 0.04
        mk = d["sm"].get(g["source"], "o")
        gc = color_by_fdr(g["nc"], FDR_CAP, "\u2193")

        ax.scatter(t, max(gr, di + 0.01), s=size_map(g["ab"]),
                   c=[gc], alpha=0.95, marker=mk,
                   edgecolors="white", linewidth=0.5, zorder=6)
        ax.plot([t, t], [max(gr, di), max(lr + 0.01, 0.06)],
                color="#AAA", lw=0.4, alpha=0.5, zorder=4)
        ad2 = np.degrees(t) % 360
        tr = ad2 - 90 if ad2 <= 180 else ad2 + 90
        h2 = "right" if ad2 <= 180 else "left"
        ax.text(t, max(lr, 0.05), g["gene"], fontsize=5, color="#333",
                fontstyle="italic", rotation=tr,
                rotation_mode="anchor", ha=h2, va="center", zorder=10)

# Axis styling
ax.set_ylim(0, do + 0.45)
ax.set_yticklabels([]); ax.set_xticklabels([])
ax.grid(False); ax.spines["polar"].set_visible(False)
ax.set_facecolor("white")

# --- Legends ---

# 1) Color gradient legends (up/down)
cax_up = fig.add_axes([0.08, 0.92, 0.12, 0.012])
norm = plt.Normalize(0, FDR_CAP)
cb_up = matplotlib.colorbar.ColorbarBase(cax_up, cmap=up_cmap, norm=norm,
                                         orientation="horizontal")
cb_up.set_label("Upregulated  -log10(FDR)", fontsize=8, labelpad=2)
cb_up.ax.tick_params(labelsize=6)

cax_dn = fig.add_axes([0.08, 0.89, 0.12, 0.012])
cb_dn = matplotlib.colorbar.ColorbarBase(cax_dn, cmap=dn_cmap, norm=norm,
                                         orientation="horizontal")
cb_dn.set_label("Downregulated  -log10(FDR)", fontsize=8, labelpad=2)
cb_dn.ax.tick_params(labelsize=6)

# 2) Marker shape legend (subtypes)
all_sm = {}
for ct in cell_types:
    if ct in all_data:
        for src, mk in all_data[ct]["sm"].items():
            all_sm[src] = mk
sh = [Line2D([0], [0], marker=all_sm[s], color="w", markerfacecolor="#888",
             markeredgecolor="#555", markersize=6, label=s)
      for s in sorted(all_sm.keys())]
leg2 = ax.legend(handles=sh, loc="upper right", bbox_to_anchor=(1.08, 1.06),
                 fontsize=7, frameon=True, fancybox=True, edgecolor="#DDD",
                 title="Subtype", title_fontsize=8,
                 ncol=2, handletextpad=0.2, columnspacing=0.5, labelspacing=0.3)

# 3) Dot size legend
size_vals = [5, 15, 30]
size_handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor="#999",
                       markersize=math.sqrt(size_map(v)), label=f"|beta|={v}")
                for v in size_vals]
leg3 = ax.legend(handles=size_handles, loc="lower right",
                 bbox_to_anchor=(1.08, -0.02),
                 fontsize=8, frameon=True, fancybox=True, edgecolor="#DDD",
                 title="|ivw_beta|", title_fontsize=8,
                 handletextpad=0.3, labelspacing=0.5)
ax.add_artist(leg2)

# Save
out_png = os.path.join(OUTPUT_DIR, "circular_volcano_npg.png")
out_pdf = os.path.join(OUTPUT_DIR, "circular_volcano_npg.pdf")
plt.savefig(out_png, dpi=DPI, bbox_inches="tight", facecolor="white")
plt.savefig(out_pdf, bbox_inches="tight", facecolor="white")
print(f"\nSaved: {out_png}\nSaved: {out_pdf}")